# Семинар 2. Feature Engineering & Feature Selection

В этом семинаре мы разберем:
- Генерацию признаков (groupby, agg, transform, pivot_table)
- Кодирование категориальных признаков
- Обработку пропущенных значений
- Масштабирование признаков
- Отбор признаков (корреляция, importance, SHAP, Boruta и др.)

In [ ]:
# Colab: install deps; locally use `uv run jupyter lab seminar.ipynb`
import sys
if "google.colab" in sys.modules:
    !pip install -q catboost shap phik boruta boostaroota category-encoders missingno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

## 1. Загрузка данных

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/a-milenkin/Competitive_Data_Science/main/data"

car_train = pd.read_csv(f"{DATA_URL}/car_train.csv")
rides_info = pd.read_csv(f"{DATA_URL}/rides_info.csv")
fix_info = pd.read_csv(f"{DATA_URL}/fix_info.csv")

### Данные по машинам и таргетам (car_train)

In [ ]:
print(car_train.shape)
car_train.hist(figsize=(25, 4), layout=(2, 5), bins=30)
car_train.sample(3)

* `car_id` - идентификатор машины
* `model` / `car_type` / `fuel_type` - марка, класс и тип топлива машины
* `car_rating` / `riders` - общий рейтинг и общее число поездок к концу 2021-го года
* `year_to_start` / `year_to_work` - год выпуска машины и начала работы в автопарке
* `target_reg` - количество дней до поломки
* `target_class` - класс поломки (всего 9 видов)

### Информация про поездки (rides_info)

In [ ]:
print(rides_info.shape)
rides_info.hist(figsize=(25, 4), layout=(2, 5), bins=40)
rides_info.sample(3)

* `user_id` / `car_id` / `ride_id` - идентификаторы водителя, машины, поездки
* `ride_date` / `rating` - дата поездки и рейтинг, поставленный водителем
* `ride_duration` / `distance` / `ride_cost` - длительность, расстояние, стоимость поездки
* `speed_avg` / `speed_max` - средняя и максимальная скорости поездки
* `stop_times` / `refueling` - количество остановок и флаг дозаправки
* `user_ride_quality` - оценка манеры вождения, определенная ML-системой сервиса
* `deviation_normal` - показатель датчиков о состоянии машины относительно нормы

### Данные про ремонт машин (fix_info)

In [ ]:
fix_info = fix_info.sort_values("worker_id")
print(fix_info.shape)
fix_info.hist(figsize=(12, 2))
fix_info.sample(3)

* `worker_id` / `car_id` - идентификатор работника и машины
* `work_type` / `work_duration` - тип и длительность (в часах) проводимой работы
* `destroy_degree` - степень износа/поврежденности машины в случае поломки
* `fix_date` - время начала ремонта (время снятия машины с линии)

## 2. Генерация признаков (Feature Engineering)

### 2.1 Как придумывать признаки

- Начать с сырых данных
- Брать все, что есть. Покрыть признаками всю имеющуюся информацию в данных
- Предполагать, от чего зависит таргет (время износа от числа поездок)
- Смотреть визуально на классы/ошибки и делать предположения. Какие полезны?
- Совсем много признаков может быть вредно. Потом придется отфильтровывать.

### 2.2 `groupby()` + `agg()` - два стиля

[`df.aggregate()` == `df.agg()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.aggregate.html) - агрегирует с помощью одной или нескольких указанных операций/функций по заданной оси.

**Способ 1** (не лучший)

In [ ]:
fix_info.groupby("car_id", as_index=False).aggregate(
    {
        "worker_id": ["count"],
        "work_duration": ["max", "mean"],
    }
).head(3)

**Способ 2** (красивый)

In [ ]:
fix_info.groupby("car_id", as_index=False).agg(
    worker_id_count=("worker_id", "count"),
    work_duration_max=("work_duration", "max"),
    work_duration_mean=("work_duration", "mean"),
).head(3)

### 2.3 `groupby()` + `transform()` - хороший vs плохой подход

[`df.transform()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.transform.html) - вызывает функцию для создания DataFrame с той же формой оси, что и у исходной таблицы.

**Способ 1** (хороший)

In [ ]:
# Среднее время выполнения работы мастером
print(fix_info.shape)

fix_info["worker_speed"] = fix_info.groupby("worker_id")["work_duration"].transform("mean")

print(fix_info.shape)
fix_info.head(3)

**Способ 2** (плохой)

In [ ]:
tmp = fix_info.groupby("worker_id", as_index=False).agg(
    work_duration_mean=("work_duration", "mean")
)

fix_info.merge(tmp, on="worker_id", how="left").head(3)

In [ ]:
fix_info["worker_experience"] = fix_info.groupby("worker_id")["car_id"].transform("count")
fix_info.head(3)

### 2.4 Пользовательские функции агрегации (lambda, nunique, quantile, mode)

Пример генерации признаков из информации про ремонт.

In [ ]:
fix_info.head(3)

In [ ]:
# число уникальных значений
f_nuniq = lambda x: x.nunique()

# число значений, которые больше чем n
more_than_n_func = lambda x, n=8: sum(x > n)

# 30% перцентиль / квантиль уровня 0.3
def quant_func(x):
    return x.quantile(0.3)

In [ ]:
# Функции поиска первой и второй моды для категориальных значений
first_mode = lambda x: x.value_counts().index[0]
second_mode = lambda x: x.value_counts().index[1]

In [ ]:
fix_info_gr = fix_info.groupby("car_id", as_index=False).agg(
    # Все встроенные статистики
    worker_count=("worker_id", "count"),
    work_duration_mean=("work_duration", "mean"),
    work_duration_max=("work_duration", "max"),
    destroy_degree_std=("destroy_degree", "std"),
    destroy_degree_sum=("destroy_degree", "sum"),
    # Самописные функции для категорий
    work_type_nuniq=("work_type", f_nuniq),
    work_type_mode=("work_type", first_mode),
    work_type_second_mode=("work_type", second_mode),
    # Самописные функции для численных
    destroy_degree_crit_q=("destroy_degree", more_than_n_func),
    worker_quant_exp=("worker_experience", quant_func),
)

fix_info_gr.sample(3)

In [ ]:
fix_info_gr.hist(figsize=(25, 9), layout=(2, 4), bins=40);

### 2.5 Проверяйте признаки!

При генерации фичей легко может пойти что-то не так.
Вы задумывали одно, а получили на деле совсем иное.

* `df.feature.hist()` - вывести гистограмму
* `df.feature.value_counts()` - вывести численное распределение

In [ ]:
bad_func = lambda x: sum(x <= -100)

tmp = fix_info.groupby("car_id", as_index=False).agg(
    good_feature=("work_duration", "mean"),
    gold_feature=("work_duration", "max"),
    killer_feature=("destroy_degree", "std"),
    bad_feature=("destroy_degree", bad_func),
)

tmp.hist(figsize=(25, 3), layout=(1, 4))
tmp.sample(3)

### 2.6 Визуальный анализ признаков (seaborn displot)

In [ ]:
# Добавим к исходной таблице новые признаки
tmp = car_train.merge(fix_info_gr, on="car_id", how="left")
tmp.head(3)

In [ ]:
g = sns.displot(
    data=tmp,
    x="destroy_degree_sum",
    y="riders",
    aspect=2,
    kind="hist",
    alpha=0.8,
    hue="target_class",
    col="work_type_second_mode",
).set_xticklabels(rotation=45, horizontalalignment="right")
1

In [ ]:
g = sns.displot(
    data=tmp,
    x="destroy_degree_std",
    y="riders",
    aspect=2,
    kind="hist",
    alpha=0.8,
    hue="target_class",
    col="work_type_second_mode",
).set_xticklabels(rotation=45, horizontalalignment="right");

Визуально видно, что `destroy_degree_std` полезнее для классификации, чем признак `destroy_degree_sum`

### 2.7 `pivot_table()` + `aggfunc()`

<img src='https://raw.githubusercontent.com/dm-fedorov/pandas_basic/master/pic/pivot_table_pandas.png' width=700>

In [ ]:
fix_info.head(3)

In [ ]:
fix_info_pivot = fix_info.pivot_table(
    index="car_id",
    columns=["work_type"],
    values=["destroy_degree"],
    aggfunc=["mean", "count"],
).fillna(0)

fix_info_pivot.columns = [f"{i[2]}_{i[0]}" for i in fix_info_pivot.columns]
fix_info_pivot.reset_index(inplace=True)

fix_info_pivot.sample(3)

### 2.8 Генерация признаков из истории поездок (временные паттерны)

In [ ]:
rides = car_train.merge(rides_info, on="car_id", how="left")
rides.head(4)

In [ ]:
# Как ведет себя deviation_normal во времени для нескольких автомобилей
cols2select = ["deviation_normal", "ride_date", "target_class", "car_id", "user_ride_quality"]
ids2select = ["f-4873956c", "p-7109749V", "p-3304414p", "f-1300760u", "L-4452446Z"]
tmp = rides[rides["car_id"].isin(rides.car_id.sample(1000, random_state=8).unique()[:15])]
tmp = tmp[tmp["car_id"].isin(ids2select)][cols2select]

In [ ]:
g = sns.relplot(
    data=tmp,
    kind="line",
    x="ride_date",
    y="deviation_normal",
    hue="target_class",
    aspect=4,
    style="car_id",
    legend=True,
)
g.set_xticklabels(rotation=45, horizontalalignment="right", step=4);

### Что можно сгенерировать через `groupby`?

Для каждой машины есть своя история поездок, у которой, в зависимости от будущей поломки, есть свои паттерны:

- Статистики (среднее/дисперсия/максимум/количество)
- Сложные признаки с помощью функций

### Покрывайте все имеющиеся признаки!

В других признаках тоже можно найти интересные паттерны.

In [ ]:
g = sns.relplot(
    data=tmp,
    kind="line",
    x="ride_date",
    y="user_ride_quality",
    hue="target_class",
    aspect=4,
    style="car_id",
    legend=True,
)
g.set_xticklabels(rotation=45, horizontalalignment="right", step=2);

Примеры признаков, которые можно сгенерировать:

* `feature_min_max_diff` : разница между max и min значениями `deviation_normal` для каждой машины
* `feature_corner` : угол наклона по признаку `user_ride_quality` для каждой машины
* `feature_mean` : среднее значение `deviation_normal` для каждой машины
* `feature_shift` : точка перегиба/сдвига для `deviation_normal`
* `feature_start` : значение точки старта для `deviation_normal`
* `feature_nans` : сумма пропусков для столбца `...` для каждой машины
* `feature_quant` : `X %` квантиль для столбца `...` для каждой машины

## 3. Кодирование категориальных признаков

Многие модели (линейные, KNN, нейросети) не умеют работать с категориальными признаками напрямую.
Нужно преобразовать их в числа. Рассмотрим основные подходы.

### 3.1 Label Encoding

Каждой категории присваивается уникальное целое число. Простой подход, но вносит ложный порядок между категориями.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
car_train["fuel_type_le"] = le.fit_transform(car_train["fuel_type"])
car_train[["fuel_type", "fuel_type_le"]].drop_duplicates()

### 3.2 One-Hot Encoding

Каждая категория превращается в отдельный бинарный столбец. Не вносит ложного порядка, но при большом количестве уникальных значений приводит к "проклятию размерности" (curse of dimensionality).

In [ ]:
ohe = pd.get_dummies(car_train[["fuel_type"]], prefix="fuel")
ohe.head()

In [ ]:
# Для столбца model OHE создаст слишком много колонок:
print(f"Уникальных моделей: {car_train['model'].nunique()}")
print("OHE для model добавит столько же колонок - не всегда оправдано.")

### 3.3 Target Encoding

Каждую категорию заменяем средним значением таргета для этой категории.

**Внимание: data leakage!** Target encoding на train без регуляризации приводит к утечке информации о таргете в признаки. Используйте его корректно (с фолдами или сглаживанием).

In [ ]:
import category_encoders as ce

te = ce.TargetEncoder(cols=["car_type"], smoothing=1.0)
car_train["car_type_te"] = te.fit_transform(
    car_train["car_type"], car_train["target_reg"]
)
car_train[["car_type", "car_type_te"]].drop_duplicates().sort_values("car_type_te")

### 3.4 Frequency / Count Encoding

Заменяем категорию на частоту (или количество) ее появления в данных. Простой и безопасный способ - нет утечки таргета.

In [ ]:
freq = car_train["model"].value_counts(normalize=True)
car_train["model_freq"] = car_train["model"].map(freq)
car_train[["model", "model_freq"]].drop_duplicates().sort_values("model_freq", ascending=False).head()

## 4. Обработка пропущенных значений

### 4.1 Диагностика пропусков

In [ ]:
# Посмотрим на пропуски в rides_info
print(rides_info.isnull().sum())
print(f"\nВсего строк: {len(rides_info)}")

In [ ]:
import missingno as msno

msno.matrix(rides_info.sample(1000, random_state=42), figsize=(12, 4))
plt.title("Паттерн пропусков в rides_info")
plt.show()

### 4.2 Стратегии заполнения

Основные подходы:

| Стратегия | Когда использовать |
|---|---|
| `dropna()` | Пропусков мало (<5%) и они случайны |
| `fillna(const)` | Пропуск имеет смысл (напр., 0 для количества) |
| `fillna(median/mean)` | Числовые признаки, пропуски случайны |
| `groupby().transform(fillna)` | Заполнение медианой внутри группы |
| `SimpleImputer` | Удобно для sklearn-пайплайнов |
| Флаг-столбец `is_missing` | Если сам факт пропуска несет информацию |

In [ ]:
# Пример: fillna медианой
rides_filled = rides_info.copy()
rides_filled["speed_max"] = rides_filled["speed_max"].fillna(rides_filled["speed_max"].median())
print(f"Пропусков в speed_max после заполнения: {rides_filled['speed_max'].isnull().sum()}")

In [ ]:
from sklearn.impute import SimpleImputer

# SimpleImputer для нескольких столбцов сразу
num_cols = ["speed_max", "rating"]
imputer = SimpleImputer(strategy="median")
rides_filled[num_cols] = imputer.fit_transform(rides_filled[num_cols])
print(rides_filled[num_cols].isnull().sum())

In [ ]:
# Заполнение медианой внутри группы + флаг пропуска
rides_filled = rides_info.copy()
rides_filled["speed_max_missing"] = rides_filled["speed_max"].isnull().astype(int)
rides_filled["speed_max"] = rides_filled.groupby("car_id")["speed_max"].transform(
    lambda x: x.fillna(x.median())
)
print(rides_filled[["speed_max", "speed_max_missing"]].head(10))

## 5. Масштабирование признаков

Масштабирование важно для моделей, чувствительных к масштабу признаков: линейных моделей, KNN, SVM, нейросетей.

Деревья (и бустинги) инвариантны к масштабированию - им все равно.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Возьмем несколько числовых столбцов для демонстрации
demo_cols = ["car_rating", "riders", "year_to_start"]
demo_data = car_train[demo_cols].copy()
demo_data.describe().round(2)

### 5.1 StandardScaler

Центрирует данные (mean=0, std=1). Самый распространенный подход.

In [ ]:
scaler = StandardScaler()
scaled = pd.DataFrame(scaler.fit_transform(demo_data), columns=demo_cols)
scaled.describe().round(2)

### 5.2 MinMaxScaler

Приводит значения к диапазону [0, 1]. Чувствителен к выбросам.

In [ ]:
scaler = MinMaxScaler()
scaled = pd.DataFrame(scaler.fit_transform(demo_data), columns=demo_cols)
scaled.describe().round(2)

### 5.3 RobustScaler

Использует медиану и IQR вместо mean/std. Устойчив к выбросам.

In [ ]:
scaler = RobustScaler()
scaled = pd.DataFrame(scaler.fit_transform(demo_data), columns=demo_cols)
scaled.describe().round(2)

### 5.4 Когда что использовать

| Скейлер | Когда использовать |
|---|---|
| `StandardScaler` | По умолчанию для линейных моделей, PCA, SVM |
| `MinMaxScaler` | Когда нужен конкретный диапазон [0,1], напр. для нейросетей |
| `RobustScaler` | Когда в данных есть выбросы |
| Без скейлинга | Деревья, бустинги (CatBoost, XGBoost, LightGBM) |

## 6. Отбор признаков (Feature Selection)

### 6.1 Зачем отбирать признаки?

Мы можем быстро нагенерировать большое количество фичей. Зачем же часть из них выкидывать?

* Если фичей очень много, данные могут не помещаться в память; существенно увеличивается время обучения модели
* С увеличением количества признаков часто падает точность предсказания модели. Особенно, если в данных большое количество мусорных фичей. Некоторые алгоритмы при сильном увеличении числа признаков вообще перестают адекватно работать - оверфит!
* Даже если точность не снижается, есть риск, что модель опирается на шумные фичи, что снизит стабильность прогноза на приватной выборке

### 6.2 Три группы методов (Filter / Embedded / Wrapper)

**Filter methods (методы фильтрации)**

Основаны на статистических методах, рассматривают каждый признак независимо. Позволяют оценить и ранжировать фичи по значимости (степени корреляции с целевой переменной). Основное преимущество - низкая цена вычислений, линейно зависящая от количества признаков. Значительно быстрее wrapper и embedded методов. Хорошо работают даже когда число признаков превышает количество примеров.

Основной недостаток - рассматривают каждый признак изолированно, поэтому не такие точные.

**Embedded methods (встроенные методы)**

Встроены прямо в процесс обучения модели. Во время обучения проводится отсев - на выходе модель знает, на какие признаки обращать внимание. Требуют меньше вычислений, чем wrapper, но больше, чем filter. Основные методы: регуляризации (LASSO, Ridge), DecisionTree, RandomForest, регуляризации в бустингах и нейросетях.

**Wrapper methods (методы-обертки)**

Оборачивают обучение модели в последовательное удаление (backward) или добавление (forward) признаков. Backward feature selection лучше отслеживает взаимосвязи между фичами, но гораздо дороже вычислительно. Основной недостаток - долгое время вычислений. Примеры: RFE (scikit-learn), Boruta, BoostARoota.

Порой бывает трудно однозначно определить к какой группе относится тот или иной метод. Например, CatBoost: feature importance с дефолтными параметрами - filter метод; с регуляризацией - гибрид; встроенная функция `select_features()` - wrapper-метод.

### 6.3 Подготовка данных

In [ ]:
# Загружаем датасет из quickstart'а
df = pd.read_csv(f"{DATA_URL}/quickstart_train.csv")
df.head(3)

In [ ]:
df.deviation_normal_count.value_counts()

Что можно сразу удалить?

* Константы
* Уникальные значения (в том числе в тесте, как правило это ID-шники типа `car_id`)

In [ ]:
cols2drop = ["car_id", "deviation_normal_count"]
df.drop(cols2drop, axis=1, inplace=True, errors="ignore")

Добавим рандомные признаки в датасет (чтобы потом проверить, что методы отбора их отсеивают).

In [ ]:
np.random.seed(42)
df["random_int"] = np.random.randint(-20, 200, df.shape[0])
df["random_num"] = np.random.random(size=df.shape[0])
df["random_norm"] = np.random.normal(loc=4, scale=1.5, size=df.shape[0])
df["random_cat"] = np.random.choice(
    ["A", "B", "C", "D"], p=[0.20, 0.3, 0.45, 0.05], size=df.shape[0]
)
df["random_ord"] = np.random.choice(
    [1, 10, 100, 1000], p=[0.40, 0.3, 0.2, 0.1], size=df.shape[0]
)

df.hist(figsize=(20, 7), layout=(-1, 5), bins=30);

### 6.4 Линейная корреляция (Pearson)

In [ ]:
corrs = df.dropna().corr(numeric_only=True).round(3).sort_values("target_reg")
sns.heatmap(corrs, cmap="Greens", square=True, vmin=0)

**Преимущества и недостатки фильтрации по корреляции:**

* [+] Быстро и понятно
* [-] Не улавливает нелинейные зависимости
* [-] Упускает парные зависимости
* [-] Не подходит для категорий (нужен другой стат. критерий)

### 6.5 PhiK корреляция

* Документация: https://pypi.org/project/phik/
* Туториал: https://towardsdatascience.com/phik-k-get-familiar-with-the-latest-correlation-coefficient-9ba0032b37e7

In [ ]:
import phik
from phik.report import plot_correlation_matrix
from phik import report

In [ ]:
phik_overview = df.phik_matrix().round(2).sort_values("target_reg")

plot_correlation_matrix(
    phik_overview.values,
    x_labels=phik_overview.columns,
    y_labels=phik_overview.index,
    vmin=0, vmax=1, color_map="Greens",
    title=r"correlation $\phi_K$",
    fontsize_factor=0.8, figsize=(11, 6),
)
plt.tight_layout()

In [ ]:
significance_overview = df.significance_matrix().fillna(0).round(1).sort_values("target_reg")

plot_correlation_matrix(
    significance_overview.values,
    x_labels=significance_overview.columns,
    y_labels=significance_overview.index,
    vmin=0, vmax=1, color_map="Greens",
    title="Significance of the coefficients",
    usetex=False, fontsize_factor=0.8, figsize=(11, 6),
)
plt.tight_layout()

**Интерпретация логарифмической вероятности**

Если логарифмическая вероятность вашего результата больше 6.63, вероятность того, что результат произойдет случайно, составляет менее 1% (p < 0.01). Если >= 3.84, то менее 5% (p < 0.05).

**Преимущества и недостатки PhiK:**

* [+] Работает с категориальными значениями!
* [+] Ловит нелинейные зависимости!
* [-] Не ловит парные зависимости
* [-] Долго считается, если много признаков

### 6.6 CatBoost Feature Importance

Для примера возьмем библиотеку градиентного бустинга CatBoost.

In [ ]:
from catboost import CatBoostRegressor, Pool, CatBoostClassifier
from sklearn.model_selection import train_test_split

In [ ]:
drop_cols = ["car_id", "target_class", "target_reg"]
cat_cols = ["car_type", "fuel_type", "model", "random_cat"]

X = df.drop(drop_cols, axis=1, errors="ignore")
y = df["target_class"].fillna(0)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
model = CatBoostClassifier(
    random_state=42,
    cat_features=cat_cols,
    thread_count=-1,
)
model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    verbose=100, plot=False,
    early_stopping_rounds=100,
)

In [ ]:
# Важность признаков CatBoost
fi = model.get_feature_importance(prettified=True)
fi

In [ ]:
feature_importance = model.feature_importances_
sorted_idx = np.argsort(feature_importance)
fig = plt.figure(figsize=(12, 6))
plt.barh(range(len(sorted_idx)), feature_importance[sorted_idx], align="center")
plt.yticks(range(len(sorted_idx)), np.array(X.columns)[sorted_idx])
plt.title("Feature Importance");

Найденный топ фичей далеко не всегда будет подмножеством, на котором модель покажет наилучшую точность. При наличии большого числа сильно скоррелированных признаков они поделят importance между собой и упадут вниз в топе по важности.

### 6.7 Permutation Importance

`Permutation Importance` из `scikit-learn` произвольным образом перетасовывает значения в одном столбце из датасета валидации, оставив остальные нетронутыми. Признак считается "важным", если точность модели падает. "Неважным" - если перетасовка не влияет на точность.

In [ ]:
from sklearn.inspection import permutation_importance

perm_importance = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=1066)
sorted_idx = perm_importance.importances_mean.argsort()
fig = plt.figure(figsize=(12, 6))
plt.barh(range(len(sorted_idx)), perm_importance.importances_mean[sorted_idx], align="center")
plt.yticks(range(len(sorted_idx)), np.array(X.columns)[sorted_idx])
plt.title("Permutation Importance");

**Можно ли рандомные признаки использовать, как границу для отсеивания?**

Да, можно, но нет гарантий, что это даст улучшение. Можно случайно выбросить полезные признаки. Надо проверять валидацией!

Переобученная модель может давать неверную важность признаков. Важность признаков зависит от распределения данных, а оно может отличаться при обучении и инференсе.

### 6.8 SHAP values

Более современный способ оценки важности признаков, основанный на теории игр. Позволяет оценить важность признаков на конкретном тестовом примере.

In [ ]:
import shap

explainer = shap.TreeExplainer(model)

val_dataset = Pool(data=X_test, label=y_test, cat_features=cat_cols)
shap_values = explainer.shap_values(val_dataset)
shap.summary_plot(shap_values, X_test, max_display=25)

### 6.9 CatBoost `select_features()` (рекурсивные методы)

Суть рекурсивных алгоритмов: удаляем признаки и смотрим, уменьшится ли качество. Если уменьшилось - значит признак был полезен.

В CatBoost есть встроенный метод [`select_features`](https://catboost.ai/en/docs/concepts/python-reference_catboost_select_features), поддерживающий 3 алгоритма:
* `RecursiveByPredictionValuesChange` - самый быстрый
* `RecursiveByLossFunctionChange` - оптимальный по соотношению точность/скорость
* `RecursiveByShapValues` (по умолчанию) - наиболее точный, но самый ресурсозатратный

In [ ]:
summary = model.select_features(
    X_train, y_train,
    eval_set=(X_test, y_test),
    features_for_select="0-13",
    num_features_to_select=8,
    steps=1,
    train_final_model=False,
    logging_level="Silent",
)

In [ ]:
# Список отобранных фичей
print(summary["selected_features_names"])
# Лучшее значение лосса
print(f"Best loss: {summary['loss_graph']['loss_values'][-1]}")

In [ ]:
# Полный отчет работы алгоритма
summary

Видим, что с каждой итерацией отбрасывалось по 1 фиче и точность каждый раз возрастала.

### 6.10 Boruta

Метод отбора признаков, пришедший из языка R. Хорошо работает только с RandomForest и достаточно долго вычисляется.

Принцип работы:
* Создается копия всех признаков; значения новых признаков случайно перемешиваются ("теневые" / shadow features)
* N раз запускается обучение случайного леса для сглаживания шума
* По вычисленному порогу часть признаков отсекается
* Новый раунд с первого пункта
* Исходные признаки попадают в 3 зоны: красную (мусор), синюю (сохраняем), зеленую (самые сильные)

In [ ]:
X_train_ohe = pd.get_dummies(X_train[cat_cols])
X_train_boruta = pd.concat(
    (X_train.drop(columns=cat_cols), X_train_ohe), axis=1
).fillna(0)
X_train_boruta.head()

In [ ]:
from boruta import BorutaPy
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_jobs=-1, max_depth=3)
boruta = BorutaPy(
    estimator=forest,
    n_estimators="auto",
    max_iter=8,
    verbose=1,
)

boruta.fit(np.array(X_train_boruta), np.array(y_train))

green_area = X_train_boruta.columns[boruta.support_].to_list()
blue_area = X_train_boruta.columns[boruta.support_weak_].to_list()
red_area = X_train_boruta.columns[~(boruta.support_ | boruta.support_weak_)].to_list()
print("features in the green area:", green_area)
print("features in the blue area:", blue_area)
print("features in the red area:", red_area)

Boruta показывает хорошие результаты со случайным лесом, но имеет недостаток - медленные вычисления, особенно на больших данных. Плохо работает на других алгоритмах (бустинг, нейросети).

### 6.11 BoostARoota

Похож на Boruta, но использует XGBoost вместо RandomForest. Требует гораздо меньше времени. Перед применением необходимо выполнить dummy-кодирование категориальных признаков.

In [ ]:
from boostaroota import BoostARoota
import boostaroota.boostaroota as _br_mod
from sklearn.preprocessing import LabelEncoder as LE

# BoostARoota not updated since 2019; fix two pandas-incompatible lines:
# 1) _create_shadow: np.random.shuffle on read-only .values (pandas >= 3.0)
# 2) _reduce_vars_xgb: df.mean(axis=1) on mixed str+float cols (pandas >= 2.0)
_orig_reduce = _br_mod._reduce_vars_xgb
import functools, operator, xgboost as xgb

def _patched_reduce_vars_xgb(x, y, metric, this_round, cutoff, n_iterations, delta, silent):
    if metric == "mlogloss":
        param = {"objective": "multi:softmax", "eval_metric": "mlogloss",
                 "num_class": len(np.unique(y))}
    else:
        param = {"eval_metric": metric}
    for i in range(1, n_iterations + 1):
        new_x, shadow_names = _br_mod._create_shadow(x)
        dtrain = xgb.DMatrix(new_x, label=y)
        bst = xgb.train(param, dtrain, verbose_eval=False)
        if i == 1:
            df = pd.DataFrame({"feature": new_x.columns})
        importance = sorted(bst.get_fscore().items(), key=operator.itemgetter(1))
        df2 = pd.DataFrame(importance, columns=["feature", f"fscore{i}"])
        df2[f"fscore{i}"] = df2[f"fscore{i}"] / df2[f"fscore{i}"].sum()
        df = pd.merge(df, df2, on="feature", how="outer")
        if not silent:
            print("Round:", this_round, "iteration:", i)
    df["Mean"] = df.select_dtypes(include="number").mean(axis=1)  # fix: skip 'feature' col
    real_vars = df[~df["feature"].isin(shadow_names)]
    shadow_vars = df[df["feature"].isin(shadow_names)]
    mean_shadow = shadow_vars["Mean"].mean() / cutoff
    real_vars = real_vars[(real_vars.Mean > mean_shadow)]
    criteria = (len(real_vars["feature"]) / len(x.columns)) > (1 - delta)
    return criteria, real_vars["feature"]

def _patched_create_shadow(x_train):
    x_shadow = x_train.copy()
    for c in x_shadow.columns:
        arr = x_shadow[c].to_numpy().copy()  # fix: writeable copy
        np.random.shuffle(arr)
        x_shadow[c] = arr
    shadow_names = ["ShadowVar" + str(i + 1) for i in range(x_train.shape[1])]
    x_shadow.columns = shadow_names
    return pd.concat([x_train, x_shadow], axis=1), shadow_names

_br_mod._create_shadow = _patched_create_shadow
_br_mod._reduce_vars_xgb = _patched_reduce_vars_xgb

le_br = LE()
y_train_enc = le_br.fit_transform(y_train)

br = BoostARoota(metric="logloss", silent=True)
br.fit(X_train_boruta, y_train_enc)

In [ ]:
# Отобранные признаки
pd.DataFrame(br.keep_vars_)

Также стоит присмотреться к библиотеке [BorutaShap](https://pypi.org/project/BorutaShap/) - wrapper-метод, сочетающий алгоритм Boruta и SHAP values.

### 6.12 Проверка качества после отбора

In [ ]:
# Возьмем топ-6 самых встречаемых в разных методах фичей
important_features = [
    "speed_max", "mean_rating", "rating_min",
    "user_uniq", "user_ride_quality_median", "car_type",
]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = CatBoostClassifier(
    random_state=42,
    cat_features=["car_type"],
    thread_count=-1,
)

model.fit(
    X_train[important_features], y_train,
    eval_set=(X_test[important_features], y_test),
    verbose=100, plot=False,
    early_stopping_rounds=100,
)

Функция ошибки снизилась. Попробуйте провести свои эксперименты и еще увеличить точность!

## 7. Итоги и советы

### Какой алгоритм отбора фичей самый лучший?

Однозначного ответа нет. Одни точные и медленные, другие слабые, но быстрые. В процессе экспериментов нужно самим найти подходящий алгоритм для конкретной задачи и данных.

### Советы по отбору признаков

* **Рекурсивные** методы отбора самые **точные**, но самые долгие (особенно когда фичей > 30)
* Перед применением алгоритмов отбора полезно проводить начальный EDA и откидывать явный мусор
* Нельзя ограничиваться каким-то одним методом
* Если не хватает ресурсов - попробуйте применить алгоритм к меньшему сэмплу данных (следите за репрезентативностью)
* Постоянно экспериментируйте и сверяйтесь с лидербордом. Всегда есть риски отсеять полезное
* Менее точными методами можно приоритизировать удаление признаков
* Методы фильтрации можно объединять между собой
* При ограниченном времени сделайте выбор в пользу **feature engineering** - новые сильные признаки могут добавить десятки % к точности, а фильтрация уже имеющихся - скорее единицы процентов

### Полезные ссылки

* [Обзорная статья по методам фильтрации](https://dataaspirant.com/feature-selection-methods-machine-learning/)
* [Библиотека scikit-feature](https://jundongl.github.io/scikit-feature/) - множество интересных методов отбора